In [1]:
import numpy as np
import scipy.io
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sub = 1
offset = 0
T_in = 1
T = 10

# --- carica ---  #
data   = scipy.io.loadmat('cylinder_wake.mat')
X_star = data['X_star']           # (N,2)
U_star = data['U_star']           # (N,2,T)
P_star = data['p_star']           # (N,T)
t_star = data['t'].squeeze()      # (T,)

# --- griglia regolare (completa) ---
dec =12
x_unique = np.unique(np.round(X_star[:,0], dec))   # 100 valori
y_unique = np.unique(np.round(X_star[:,1], dec))   # 50 valori
Sx_full, Sy_full = len(x_unique), len(y_unique)
time = t_star.shape[0]

# mapping (x,y) -> (i,j) sulla griglia COMPLETA
x_to_i = {x:i for i,x in enumerate(x_unique)}
y_to_j = {y:j for j,y in enumerate(y_unique)}
ij = np.empty((X_star.shape[0],2), dtype=int)
for k,(x,y) in enumerate(np.round(X_star, dec)):
    ij[k] = (x_to_i[x], y_to_j[y])

# --- riempi griglie complete ---
U_grid = np.empty((Sx_full, Sy_full, time, 2), dtype=U_star.dtype)  # ...,(u,v)
P_grid = np.empty((Sx_full, Sy_full, time),     dtype=P_star.dtype)

for k in range(X_star.shape[0]):
    i,j = ij[k]
    U_grid[i,j,:,0] = U_star[k,0,:]   # u
    U_grid[i,j,:,1] = U_star[k,1,:]   # v
    P_grid[i,j,:]   = P_star[k,:]

# ====== QUI scegli metà dominio in x (50 colonne) ======
half = Sx_full // 2

# Metà SINISTRA:    ix = slice(0, half)
# Metà DESTRA:      ix = slice(half, Sx_full)
# Metà CENTRATA:    c = Sx_full//2; w = half; ix = slice(c - w//2, c + w//2)
ix = slice(0, half)

# Integra anche l'eventuale subsampling
iy = slice(0, Sy_full)               # tieni tutte le 50 in y
ix = slice(ix.start, ix.stop, sub)   # es. sub=1 → tutte; sub=2 → una ogni 2
iy = slice(iy.start, iy.stop, sub)

# --- in torch, con B=1: u,v,p hanno shape (1, Sx', Sy', T) ---
u = torch.from_numpy(U_grid[ix, iy, :, 0]).unsqueeze(0).contiguous()   # (1, 50, 50, T) se sub=1
v = torch.from_numpy(U_grid[ix, iy, :, 1]).unsqueeze(0).contiguous()
p = torch.from_numpy(P_grid[ix, iy, :]).unsqueeze(0).contiguous()

# --- coord grid coerenti (usando meshgrid) ---
x_half = x_unique[ix]
y_half = y_unique[iy]
Xv, Yv = np.meshgrid(x_half, y_half, indexing='ij')  # (Sx',Sy')
X = torch.from_numpy(Xv)[None, ..., None]  # (1,Sx',Sy',1)
Y = torch.from_numpy(Yv)[None, ..., None]  # (1,Sx',Sy',1)

# --- split in/stimolo e target ---
u_in = u[..., offset:offset+T_in]                         # (1,Sx',Sy',T_in)
v_in = v[..., offset:offset+T_in]
p_in = p[..., offset:offset+T_in]

u_t = u[..., offset+T_in:offset+T_in+T]                   # (1,Sx',Sy',T)
v_t = v[..., offset+T_in:offset+T_in+T]
p_t = p[..., offset+T_in:offset+T_in+T]

# pack finale (B=1): concateni (x,y) ai canali temporali
train_in = torch.cat([X.to(device), Y.to(device), 
                      u_in.to(device), v_in.to(device), p_in.to(device)], dim=-1)  # (1,Sx',Sy', 2+3*T_in)

target = torch.stack([u_t, v_t, p_t], dim=-1)  # (1,Sx',Sy', T, 3)
train_t = target.reshape(target.shape[0], target.shape[1], target.shape[2], -1).contiguous().to(device)

print("Shapes ->", "train_in:", tuple(train_in.shape), "train_t:", tuple(train_t.shape),
      "| sub =", sub, "| half-x =", (ix.start, ix.stop, ix.step))

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_in, train_t),
    batch_size=1, shuffle=False
)

Shapes -> train_in: (1, 50, 50, 5) train_t: (1, 50, 50, 30) | sub = 1 | half-x = (0, 50, 1)


In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from utilities3 import *

import operator
from functools import reduce
from functools import partial
import os

from timeit import default_timer
import scipy.io
import pandas as pd
from get_params import get_params

import math


torch.manual_seed(0)
np.random.seed(0)

################################################################
# fourier layer
################################################################

def compl_mul2d_rfft(a, b):
    # a: (batch, in_ch, m1, m2, 2)
    # b: (in_ch, out_ch, m1, m2, 2)
    op = partial(torch.einsum, "bctq,dctq->bdtq")
    real = op(a[...,0], b[...,0]) - op(a[...,1], b[...,1])
    imag = op(a[...,1], b[...,0]) + op(a[...,0], b[...,1])
    return torch.stack([real, imag], dim=-1)


class SpectralConv2d_fast(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1/(in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, 2))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, 2))

    def forward(self, x):
        batchsize, _, nx, ny = *x.shape, 
        # Compute Fourier coefficients (real input -> rfft2)
        x_ft_complex = torch.fft.rfft2(x, norm='ortho')
        # Convert to real-imag tensor
        x_ft = torch.stack([x_ft_complex.real, x_ft_complex.imag], dim=-1)
        # Allocate output in Fourier space
        out_ft = torch.zeros(batchsize, self.out_channels, nx, ny//2+1, 2, device=x.device)
        # Multiply modes
        out_ft[:, :, :self.modes1, :self.modes2] = \
            compl_mul2d_rfft(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            compl_mul2d_rfft(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        # Convert back to complex
        out_ft_complex = torch.complex(out_ft[...,0], out_ft[...,1])
        # Return to physical space
        x = torch.fft.irfft2(out_ft_complex, s=(nx, ny), norm='ortho')
        return x

class SimpleBlock2d(nn.Module):
    def __init__(self, modes1, modes2, width):
        super(SimpleBlock2d, self).__init__()

        self.modes1 = modes1
        self.modes2 = modes2
        self.width = width
        self.fc0 = nn.Linear(5, self.width)

        self.conv0 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv1 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv2 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv3 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)
        self.bn0 = torch.nn.BatchNorm2d(self.width)
        self.bn1 = torch.nn.BatchNorm2d(self.width)
        self.bn2 = torch.nn.BatchNorm2d(self.width)
        self.bn3 = torch.nn.BatchNorm2d(self.width)


        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        batchsize = x.shape[0]
        size_x, size_y = x.shape[1], x.shape[2]

        x = self.fc0(x)
        x = x.permute(0, 3, 1, 2)

        x1 = self.conv0(x)
        x2 = self.w0(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn0(x1 + x2)
        x = F.relu(x)
        x1 = self.conv1(x)
        x2 = self.w1(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn1(x1 + x2)
        x = F.relu(x)
        x1 = self.conv2(x)
        x2 = self.w2(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn2(x1 + x2)
        x = F.relu(x)
        x1 = self.conv3(x)
        x2 = self.w3(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn3(x1 + x2)


        x = x.permute(0, 2, 3, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

class Net2d(nn.Module):
    def __init__(self, modes, width):
        super(Net2d, self).__init__()

        self.conv1 = SimpleBlock2d(modes, modes, width)


    def forward(self, x):
        x = self.conv1(x)
        return x


    def count_params(self):
        c = 0
        for p in self.parameters():
            c += reduce(operator.mul, list(p.size()))

        return c



In [3]:
################################################################
# training and evaluation
################################################################
# Parametri\ 

ntrain      = 1

modes       = 24
width       = 64


batch_size  = 1
epochs      = 100

learning_rate   = 2e-3
scheduler_step  = 20
scheduler_gamma = 0.5


step        = 1

rf          = 8
orders_v    = [2,2]
n_samples   = 4
model = Net2d(modes, width).cuda()
# model = torch.load('model/ns_fourier_V100_N1000_ep100_m8_w10')

print(model.count_params())
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step, gamma=scheduler_gamma)


myloss = LpLoss(size_average=False)

test_full = []

path = f'Supervised_FNO{ntrain}_Adam_ep{epochs}_m{modes}_w{width}_T{T}'

path_model     = os.path.join('model',  path)
path_image     = os.path.join('image',  path)
path_loss_dir  = os.path.join('loss',   path)
os.makedirs(path_image, exist_ok=True)
os.makedirs(path_loss_dir, exist_ok=True)
os.makedirs(os.path.dirname(path_model), exist_ok=True)

# ============================
# Training loop robusto a T lunghi
# ============================

import numpy as np
import torch

# ---- impostazioni autoregressione / stabilità ----
predict_delta   = False     # True: modello predice Δ; False: predice stato assoluto
use_detach      = True     # stacca il gradiente quando reinserisci la predizione in finestra
time_weighting  = True     # pesa di più gli step temporali lontani
noise_std       = 0.0      # es. 0.01 se lavori con canali normalizzati (std~1). 0.0 = off
grad_clip       = 1.0      # clipping del gradiente (None o 0.0 per disattivare)

# ---- scheduled sampling con "pavimento" ----
tf_max = 1.0
tf_min = 0.10                   # non scendere mai sotto il 10% di teacher forcing
warm   = int(0.6 * 200)      # epoche su cui far decrescere p GT -> tf_min

def get_tf_ratio(ep):
    # Cosine decay da 1.0 a tf_min durante il warmup
    x = min(1.0, ep / max(1, warm))
    # 0 -> 1: cos decresce da 1 a -1
    return float(tf_min + 0.5*(tf_max - tf_min)*(1.0 + torch.cos(torch.tensor(np.pi*x)).item()))

def make_boundary_mask(Sx, Sy, thickness=1, device='cuda'):
    bm = torch.zeros(1, Sx, Sy, 1, device=device)
    t = int(thickness)
    bm[:, :t, :, :]  = 1;  bm[:, -t:, :, :] = 1
    bm[:, :, :t, :]  = 1;  bm[:, :, -t:, :] = 1
    return bm  # shape (1,Sx,Sy,1)

C = 3  # (u,v,p) per passo


37774979


In [4]:
lambda_res = 1.0      # peso della residual loss (tuning a piacere)
beta_div   = 1.0      # peso del vincolo di incomprimibilità
m_fc       = 2       # spessore banda Fourier continuation
epochs = 250
C = 3

loss_full = []
# usa le epoche della fase
for ep in range(epochs):
    model.train()
    t_start = default_timer()

    # accumulatori epoca
    train_l2_step = 0.0           # loss totale per step (tutte le componenti)
    train_l2_full = 0.0           # L2 full-sequence (monitoring)


    for xx, yy in train_loader:
        xx = xx.to(device, dtype=torch.float32)  # (B,Sx,Sy, 2 + 3*T_in)
        yy = yy.to(device, dtype=torch.float32)  # (B,Sx,Sy, 3*T)

        B, Sx, Sy = xx.shape[0], xx.shape[1], xx.shape[2]

        loss = 0.0
        preds = []

        for t in range(T):
            # target (u_{t+1}, v_{t+1}, p_{t+1})
            y  = yy[..., C*t : C*(t+1)]                          # (B,Sx,Sy,3)

            # predizione
            if predict_delta:
                base = xx[..., -C:]                              # (u_t,v_t,p_t)
                delta = model(xx)                                # (Δu,Δv,Δp)
                im = base + delta                                # (u_{t+1}, v_{t+1}, p_{t+1})
            else:
                im = model(xx)

            preds.append(im)

            # ====== RESIDUAL NS (valutata su stato t) ======
            state_t = xx[..., -C:]                        # (u_t, v_t, p_t)
            u_t = state_t[..., 0]
            v_t = state_t[..., 1]
            p_t = state_t[..., 2]

            loss += myloss(im.reshape(B, -1), y.reshape(B, -1))

            xy   = xx[..., :2]
            hist = xx[..., 2:]
            hist = torch.cat([hist[..., C:], y], dim=-1)   # sposta la finestra e aggiunge feed
            xx   = torch.cat([xy, hist], dim=-1)


        # ---- metriche full-sequence ----
        pred_seq = torch.cat(preds, dim=-1)                        # (B,Sx,Sy, 3*T)
        train_l2_full += myloss(pred_seq.reshape(B, -1), yy.reshape(B, -1)).item()

        # ---- accumula su epoca ----
        train_l2_step += loss.item()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip and grad_clip > 0.0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

    scheduler.step()
    elapsed = default_timer() - t_start
    denom_steps = T * len(train_loader)

    print(f"Epoch: {ep}/150│ "
            f"Time: {elapsed:6.2f}s │ "
            f"Train step TOTAL: {train_l2_step/denom_steps:8.4e} │ "
            f"Train full L2: {train_l2_full/len(train_loader):8.4e} │ ")

    # salva serie per plotting
    loss_full.append(train_l2_full/len(train_loader))                     # solo (u,v) bordo

idx = list(range(1, epochs+1))

df = pd.DataFrame({"epoch": range(1, len(loss_full)+1),
                   "loss_full": loss_full})
df.index.name = 'epoch_in_phase'
csv_path = os.path.join(path_loss_dir, f"loss_supervised.csv")
df.to_csv(csv_path)
print(f"→ Saved loss history' in {csv_path}")
print("Training finished.")


# Save final model
torch.save(model, path_model)


Epoch: 0/150│ Time:   0.48s │ Train step TOTAL: 1.0763e+00 │ Train full L2: 1.0763e+00 │ 
Epoch: 1/150│ Time:   0.17s │ Train step TOTAL: 7.2351e-01 │ Train full L2: 7.2355e-01 │ 
Epoch: 2/150│ Time:   0.19s │ Train step TOTAL: 4.1002e-01 │ Train full L2: 4.1010e-01 │ 
Epoch: 3/150│ Time:   0.20s │ Train step TOTAL: 2.2212e-01 │ Train full L2: 2.2291e-01 │ 
Epoch: 4/150│ Time:   0.19s │ Train step TOTAL: 2.0760e-01 │ Train full L2: 2.0815e-01 │ 
Epoch: 5/150│ Time:   0.19s │ Train step TOTAL: 1.7710e-01 │ Train full L2: 1.7799e-01 │ 
Epoch: 6/150│ Time:   0.19s │ Train step TOTAL: 1.2878e-01 │ Train full L2: 1.3005e-01 │ 
Epoch: 7/150│ Time:   0.19s │ Train step TOTAL: 1.2755e-01 │ Train full L2: 1.2895e-01 │ 
Epoch: 8/150│ Time:   0.19s │ Train step TOTAL: 1.2503e-01 │ Train full L2: 1.2642e-01 │ 
Epoch: 9/150│ Time:   0.20s │ Train step TOTAL: 1.0186e-01 │ Train full L2: 1.0405e-01 │ 
Epoch: 10/150│ Time:   0.19s │ Train step TOTAL: 9.7034e-02 │ Train full L2: 9.9103e-02 │ 
Epoch: 11

In [5]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# -----------------------------------------
# genera predizioni autoregressive dal modello
# -----------------------------------------
def predict_sequence(model, xx, yy, T, step=1, C=3):
    model.eval()
    preds = []
    with torch.no_grad():
        for t in range(0, T, step):
            y = yy[..., 3*t:3*(t+1)] 
           # ---- predizione ----
            if predict_delta:
                # base = ultimo stato nella finestra (u_t, v_t, p_t)
                base = xx[..., -C:]                                # (B,Sx,Sy,3)
                delta = model(xx)                                  # (B,Sx,Sy,3)
                im = base + delta                                  # pred stato t+1
            else:
                im = model(xx)                                     # pred stato t+1 diretto
            
            preds.append(im)
            
            # aggiorna input con autoregressione
            xy   = xx[..., :2]
            hist = xx[..., 2:]
            hist = torch.cat([hist[..., C:], y], dim=-1)
            xx   = torch.cat([xy, hist], dim=-1)
    pred_seq = torch.cat(preds, dim=-1)  # (B, Sx, Sy, 3*T)
    return pred_seq


def animate_prediction(xx, yy, model, T, step=1, interval=200,
                       save_prefix="anim",  # solo basename del file (niente path)
                       cmap_field="jet", cmap_err="magma",
                       add_colorbars=True, times=None,
                       out_dir=".", save_last_png=True, filename_base=None):
    """
    out_dir: cartella dove salvare i file (es. path_image)
    save_last_png: se True salva anche l'ultimo frame come PNG
    filename_base: basename per i file; se None usa save_prefix
    """
    import os
    if filename_base is None:
        filename_base = save_prefix
    os.makedirs(out_dir, exist_ok=True)

    C = 3
    pred_seq = predict_sequence(model, xx.clone(), yy.clone(), T, step, C)  # (B,Sx,Sy,3*Tpred)
    B, Sx, Sy, _ = pred_seq.shape
    Tpred = pred_seq.shape[-1] // C

    # estrai pred e gt (B,Sx,Sy,T)
    u_pred = pred_seq[..., 0::C]; v_pred = pred_seq[..., 1::C]; p_pred = pred_seq[..., 2::C]
    u_true = yy      [..., 0::C]; v_true = yy      [..., 1::C]; p_true = yy      [..., 2::C]

    # to numpy (Sx,Sy,T)
    u_pred = u_pred.squeeze(0).detach().cpu().numpy()
    v_pred = v_pred.squeeze(0).detach().cpu().numpy()
    p_pred = p_pred.squeeze(0).detach().cpu().numpy()
    u_true = u_true.squeeze(0).detach().cpu().numpy()
    v_true = v_true.squeeze(0).detach().cpu().numpy()
    p_true = p_true.squeeze(0).detach().cpu().numpy()

    # errori (Sx,Sy,T)
    eu = np.abs(u_pred - u_true)
    ev = np.abs(v_pred - v_true)
    ep = np.abs(p_pred - p_true)

    # range colori stabile (dai GT)
    vmin_u, vmax_u = float(u_true.min()), float(u_true.max())
    vmin_v, vmax_v = float(v_true.min()), float(v_true.max())
    vmin_p, vmax_p = float(p_true.min()), float(p_true.max())

    # range errore robusto (99° percentile)
    emax_u = max(1e-12, float(np.percentile(eu, 99)))
    emax_v = max(1e-12, float(np.percentile(ev, 99)))
    emax_p = max(1e-12, float(np.percentile(ep, 99)))

    # figura animazione 3x3
    fig, axes = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
    titles_top = ['u pred', 'v pred', 'p pred']
    titles_mid = ['u GT',   'v GT',   'p GT'  ]
    titles_bot = ['|u err|','|v err|','|p err|']

    for i in range(3):
        axes[0,i].set_title(titles_top[i])
        axes[1,i].set_title(titles_mid[i])
        axes[2,i].set_title(titles_bot[i])
        for r in range(3):
            axes[r,i].set_xticks([]); axes[r,i].set_yticks([])

    k0 = 0
    im00 = axes[0,0].imshow(u_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u, animated=True)
    im01 = axes[0,1].imshow(v_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v, animated=True)
    im02 = axes[0,2].imshow(p_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p, animated=True)

    im10 = axes[1,0].imshow(u_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u, animated=True)
    im11 = axes[1,1].imshow(v_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v, animated=True)
    im12 = axes[1,2].imshow(p_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p, animated=True)

    im20 = axes[2,0].imshow(eu[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_u, animated=True)
    im21 = axes[2,1].imshow(ev[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_v, animated=True)
    im22 = axes[2,2].imshow(ep[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_p, animated=True)

    ims = [im00, im01, im02, im10, im11, im12, im20, im21, im22]
    cbs = []
    if add_colorbars:
        for ax, im in zip(axes[0], ims[0:3]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))
        for ax, im in zip(axes[1], ims[3:6]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))
        for ax, im in zip(axes[2], ims[6:9]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))

    # testo tempo/frame
    if times is not None and len(times) >= Tpred:
        time_text = fig.text(0.01, 0.98, f"t = {float(times[0]):.3f}", fontsize=12)
    else:
        time_text = fig.text(0.01, 0.98, f"frame 0 / {Tpred-1}", fontsize=12)

    def update(frame):
        im00.set_array(u_pred[...,frame].T); im01.set_array(v_pred[...,frame].T); im02.set_array(p_pred[...,frame].T)
        im10.set_array(u_true[...,frame].T); im11.set_array(v_true[...,frame].T); im12.set_array(p_true[...,frame].T)
        im20.set_array(eu[...,frame].T);     im21.set_array(ev[...,frame].T);     im22.set_array(ep[...,frame].T)
        if times is not None and len(times) >= Tpred:
            time_text.set_text(f"t = {float(times[frame]):.3f}")
        else:
            time_text.set_text(f"frame {frame} / {Tpred-1}")
        return ims + [time_text]

    ani = animation.FuncAnimation(fig, update, frames=Tpred, interval=interval, blit=False)

    # ---- salva GIF ----
    gif_path = os.path.join(out_dir, f"{filename_base}.gif")
    ani.save(gif_path, writer='pillow', fps=max(1, int(1000/interval)))

    # ---- salva PNG dell'ultimo frame (stessa griglia 3x3) ----
    png_path = None
    if save_last_png:
        frame = Tpred-1
        fig_last, axes_last = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
        titles_top = ['u pred', 'v pred', 'p pred']
        titles_mid = ['u GT',   'v GT',   'p GT'  ]
        titles_bot = ['|u err|','|v err|','|p err|']
        for i in range(3):
            axes_last[0,i].set_title(titles_top[i])
            axes_last[1,i].set_title(titles_mid[i])
            axes_last[2,i].set_title(titles_bot[i])
            for r in range(3):
                axes_last[r,i].set_xticks([]); axes_last[r,i].set_yticks([])

        axes_last[0,0].imshow(u_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u)
        axes_last[0,1].imshow(v_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v)
        axes_last[0,2].imshow(p_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p)

        axes_last[1,0].imshow(u_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u)
        axes_last[1,1].imshow(v_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v)
        axes_last[1,2].imshow(p_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p)

        axes_last[2,0].imshow(eu[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_u)
        axes_last[2,1].imshow(ev[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_v)
        axes_last[2,2].imshow(ep[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_p)

        if add_colorbars:
            for ax in axes_last[0]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)
            for ax in axes_last[1]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)
            for ax in axes_last[2]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)

        png_path = os.path.join(out_dir, f"{filename_base}_last.png")
        fig_last.savefig(png_path, dpi=160, bbox_inches="tight")
        plt.close(fig_last)

    plt.close(fig)
    return ani, gif_path, png_path



# batch di esempio
xx, yy = next(iter(train_loader))
xx = xx.to(device, dtype=torch.float32)
yy = yy.to(device, dtype=torch.float32)

# salva GIF + ultimo frame PNG nella cartella immagini dell'esperimento
ani, gif_path, png_path = animate_prediction(
    xx, yy, model,
    T=T, step=1, interval=200,
    save_prefix="supervised_fno",
    out_dir=path_image,          # <--- usa la cartella 'image/<path>'
    save_last_png=True,
)

print("GIF salvata in:", gif_path)
print("PNG ultimo frame in:", png_path)



GIF salvata in: image/Supervised_FNO1_Adam_ep100_m24_w64_T10/supervised_fno.gif
PNG ultimo frame in: image/Supervised_FNO1_Adam_ep100_m24_w64_T10/supervised_fno_last.png
